In [1]:
import sys
sys.path.append('..')
from src.train import get_mlflow_client, train_model, evaluate_model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_sample_weight
import mlflow

In [2]:
data = pd.read_csv('data/simulated_data_final.csv')

In [3]:
data.head()

,timestamp,userDataReadIops,userDataWriteIops,readLatencyMs,writeLatencyMs,cpuPercent,memoryPercent,label,iops_data_read_rolling_mean_5m,iops_data_read_rolling_std_5m,...,iops_data_write_rolling_mean_5m,iops_data_write_rolling_std_5m,iops_data_write_pct_change_1h,read_latency_rolling_mean_5m,read_latency_rolling_std_5m,read_latency_pct_change_1h,write_latency_rolling_mean_5m,write_latency_rolling_std_5m,write_latency_pct_change_1h,group
0,2026-01-01 01:00:00,3938.239326,2441.417708,2.225206,3.159634,39.301569,58.212759,0,4019.038969,144.643824,...,2337.333695,144.681146,0.062305,2.146060,0.693686,0.023486,3.077954,0.473671,0.572181,1
1,2026-01-01 01:01:00,3998.080881,2456.902445,1.735196,3.452344,41.392022,57.956846,0,4046.282991,118.015268,...,2351.826919,153.914178,0.043634,2.014212,0.697317,-0.189790,3.170657,0.496667,0.396293,1
2,2026-01-01 01:02:00,3815.102552,2554.658290,1.626288,3.075471,34.070572,52.397814,0,4015.518178,156.874498,...,2429.525350,133.457582,0.105714,1.828592,0.638636,0.061726,3.251690,0.422076,0.136334,1
3,2026-01-01 01:03:00,3798.303439,2689.779576,2.280207,3.310007,30.144235,57.683916,0,3955.557857,173.798651,...,2469.317847,178.397082,0.113036,1.760520,0.544286,-0.004186,3.368887,0.303679,0.076483,1
4,2026-01-01 01:04:00,4201.243469,2218.541534,1.134852,3.208689,38.933098,64.804512,0,3950.193933,163.387840,...,2472.259911,172.962717,-0.139990,1.800350,0.471221,-0.095705,3.241229,0.145338,-0.086385,1


In [4]:
data = data.drop(columns=['timestamp','group',])

In [5]:
split_ratio = 0.8
split_index = int(len(data) * split_ratio)
train_data = data.iloc[:split_index]
test_data = data.iloc[split_index:]

In [6]:
train_x = train_data.drop(columns=['label'])
train_y = train_data['label']
test_x = test_data.drop(columns=['label'])
test_y = test_data['label']

In [7]:
len(train_x), len(test_x)

(7952, 1988)

In [8]:
mlflow_client,experiment_id = get_mlflow_client("http://127.0.0.1:5000","xgboost_training")
print(experiment_id)
with mlflow.start_run(experiment_id=experiment_id) as run:
    mlflow.log_param("model_type", "xgboost")
    params={
        'objective': 'multi:softprob',
        'num_class': 5,
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'eval_metric': 'mlogloss'
    }
    mlflow.log_params(params)
    sample_weights = compute_sample_weight(class_weight='balanced', y=train_y)
    model =train_model(model_type="xgboost", X_train=train_x, y_train=train_y, params=params, sample_weights=sample_weights)
    metrics = evaluate_model(model, test_x, test_y)
    mlflow.log_metric("accuracy", metrics['accuracy'])
    mlflow.log_metric("f1_score", metrics['f1_score'])
    with open("classification_report.txt", "w") as f:
        f.write(str(metrics['classification_report']))
    mlflow.log_artifact("classification_report.txt")
    mlflow.xgboost.log_model(model, artifact_path="model")

2


2026/06/04 18:38:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run zealous-goat-111 at: http://127.0.0.1:5000/#/experiments/2/runs/1bba3b2e3e18490d89f60997db3d8697
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [9]:
run_id = run.info.run_id
model_version = mlflow.register_model(f"runs:/{run_id}/model", "xgboost_tsd_model")
print(f"Model registered with version: {model_version}")
mlflow_client.set_registered_model_alias(name="xgboost_tsd_model", version=model_version.version, alias="champion")

Registered model 'xgboost_tsd_model' already exists. Creating a new version of this model...
2026/06/04 18:38:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 1bba3b2e3e18490d89f60997db3d8697 has no artifacts at artifact path 'model', registering model based on models:/m-2541cc4bad6a4823b86d39ccdb1c1343 instead
2026/06/04 18:38:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost_tsd_model, version 2


Model registered with version: <ModelVersion: aliases=[], creation_timestamp=1780578518785, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1780578518785, metrics=None, model_id=None, name='xgboost_tsd_model', params=None, run_id='1bba3b2e3e18490d89f60997db3d8697', run_link='', source='models:/m-2541cc4bad6a4823b86d39ccdb1c1343', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>


Created version '2' of model 'xgboost_tsd_model'.


In [10]:
mlflow_client,experiment_id = get_mlflow_client("http://127.0.0.1:5000","isolation_forest_training")
print(experiment_id)
with mlflow.start_run(experiment_id=experiment_id) as run:
    mlflow.log_param("model_type", "isolation_forest")
    params={
        'n_estimators': 100,
        'max_samples': 'auto',
        'contamination': 0.1,
        'random_state': 42
    }

    mlflow.log_params(params)
    model =train_model(model_type="isolation_forest", X_train=train_x, params=params)
    metrics = evaluate_model(model, test_x, test_y)
    mlflow.log_metric("accuracy", metrics['accuracy'])
    mlflow.log_metric("f1_score", metrics['f1_score'])
    with open("classification_report.txt", "w") as f:
        f.write(str(metrics['classification_report']))
    mlflow.log_artifact("classification_report.txt")
    mlflow.sklearn.log_model(model, artifact_path="model")

3


2026/06/04 18:39:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 18:39:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run gaudy-squirrel-430 at: http://127.0.0.1:5000/#/experiments/3/runs/d25f80266555410fbcf4b341c5351362
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [11]:
run_id = run.info.run_id
model_version = mlflow.register_model(f"runs:/{run_id}/model", "isolation_forest_tsd_model")
print(f"Model registered with version: {model_version}")
mlflow_client.set_registered_model_alias(name="isolation_forest_tsd_model", version=model_version.version, alias="champion")

Successfully registered model 'isolation_forest_tsd_model'.
2026/06/04 18:40:10 WARNING mlflow.tracking._model_registry.fluent: Run with id d25f80266555410fbcf4b341c5351362 has no artifacts at artifact path 'model', registering model based on models:/m-873035cc0ced4671adb67e9be49f02b4 instead
2026/06/04 18:40:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: isolation_forest_tsd_model, version 1


Model registered with version: <ModelVersion: aliases=[], creation_timestamp=1780578610682, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1780578610682, metrics=None, model_id=None, name='isolation_forest_tsd_model', params=None, run_id='d25f80266555410fbcf4b341c5351362', run_link='', source='models:/m-873035cc0ced4671adb67e9be49f02b4', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>


Created version '1' of model 'isolation_forest_tsd_model'.
